In [1]:
import pandas as pd
import time
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline

In [2]:
from pathlib import Path

BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"

In [3]:
df = pd.read_csv(f'{DATA_DIR}/v1/function_dataset_cleaned.csv')

In [4]:
# 定義特徵與目標變數
X = df.drop(columns=['label'])
y = df['label']

In [5]:
# 將數據分割為訓練集與測試集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10, stratify=y)

In [6]:
print(y_train.value_counts())       # 訓練集類別

label
cubic          400
cosine         400
exponential    400
quadratic      400
reciprocal     400
sine           400
logarithmic    400
linear         400
Name: count, dtype: int64


In [7]:
print(y_test.value_counts())        # 測試集類別

label
quadratic      100
logarithmic    100
reciprocal     100
sine           100
cosine         100
exponential    100
cubic          100
linear         100
Name: count, dtype: int64


## 開始訓練模型

In [8]:
# 建立pipeline, 將數據標準化, 並蒐集模型
models = {"LogisticRegression": make_pipeline(StandardScaler(),
                                              LogisticRegression(max_iter=1000)),
          "DecisionTree": DecisionTreeClassifier(random_state=10),
          
          "RandomForest": RandomForestClassifier(random_state=10),
          
          "KNN": make_pipeline(StandardScaler(),
                               KNeighborsClassifier(n_neighbors=3)),
          "SVM": make_pipeline(StandardScaler(),
                               SVC())
         }

In [9]:
model_times = []
for name, model in models.items():
    start_time = time.perf_counter()        # 開始計時
    model.fit(X_train, y_train)
    end_time = time.perf_counter()          # 結束計時
    elapsed_time = end_time - start_time    # 計算經過時間
    model_times.append(elapsed_time)

    # 儲存模型
    joblib.dump(model, f'{MODEL_DIR}/v1/{name}_baseline.joblib')

In [10]:
print(model_times)

[0.2207641999993939, 0.3532169999962207, 2.108725300000515, 0.013007500005187467, 0.3934406000043964]


## 預測模型

In [11]:
from sklearn.metrics import *

In [22]:
results = []
trained_models = {}
time_index = 0
for name, model in models.items():
    model = joblib.load(f'{MODEL_DIR}/v1/{name}_baseline.joblib')
    trained_models[name] = model        # 儲存模型物件
    
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')

    print(f"正在預測 {name} 模型:")
    print(f"準確度: {acc:.4f}")
    print(f"混淆矩陣:\n{confusion_matrix(y_test, y_pred)}")
    print(f"分類報告:\n{classification_report(y_test, y_pred)}\n")
    results.append({'Model': name, 
                    'Accuracy': acc, 
                    'Precision': pre, 
                    'Recall': recall, 
                    'F1': f1,
                    'Time': model_times[time_index]})
    time_index += 1

正在預測 LogisticRegression 模型:
準確度: 0.1837
混淆矩陣:
[[25  5  2  1 22 11  1 33]
 [18 23 24  4 10 12  0  9]
 [ 7 13 18  1 25 22  1 13]
 [ 0  1  0  1 26 38  0 34]
 [ 7  0  1  1 24 34  0 33]
 [12 10  3  0 27 26  0 22]
 [ 0  0  0  2 38  0  9 51]
 [32 11  1  1 27  5  2 21]]
分類報告:
              precision    recall  f1-score   support

      cosine       0.25      0.25      0.25       100
       cubic       0.37      0.23      0.28       100
 exponential       0.37      0.18      0.24       100
      linear       0.09      0.01      0.02       100
 logarithmic       0.12      0.24      0.16       100
   quadratic       0.18      0.26      0.21       100
  reciprocal       0.69      0.09      0.16       100
        sine       0.10      0.21      0.13       100

    accuracy                           0.18       800
   macro avg       0.27      0.18      0.18       800
weighted avg       0.27      0.18      0.18       800


正在預測 DecisionTree 模型:
準確度: 0.6200
混淆矩陣:
[[40  0  2  3  4  1 14 36]
 [ 0 95  4  

In [23]:
results_df = pd.DataFrame(results)

In [24]:
# 儲存成網頁
results_df.to_html(f'{DATA_DIR}/v1/model_results.html')

## 模型比較

In [25]:
print(results_df.to_string(index=False, formatters={'Accuracy': '{:.4f}'.format, 
                                                    'Precision': '{:.4f}'.format, 
                                                    'Recall': '{:.4f}'.format, 
                                                    'F1': '{:.4f}'.format,
                                                    'Time': '{:.4f}'.format}))

             Model Accuracy Precision Recall     F1   Time
LogisticRegression   0.1837    0.2696 0.1837 0.1816 0.2208
      DecisionTree   0.6200    0.6139 0.6200 0.6138 0.3532
      RandomForest   0.6963    0.6952 0.6963 0.6933 2.1087
               KNN   0.6388    0.6318 0.6388 0.6307 0.0130
               SVM   0.3588    0.4527 0.3588 0.3506 0.3934


### 找出 F1 最高的模型

In [26]:
best_result = results_df.loc[results_df['F1'].idxmax()]
best_model_name = best_result['Model']
best_model = trained_models[best_model_name]
print(f"最佳模型:\n{best_result}")

最佳模型:
Model        RandomForest
Accuracy          0.69625
Precision        0.695173
Recall            0.69625
F1               0.693315
Time             2.108725
Name: 2, dtype: object


In [27]:
joblib.dump(best_model, f'{MODEL_DIR}/v1/best_model.joblib')

['d:\\Python\\我的AI作品集\\專案1_數學函數辨識\\Package\\models/v1/best_model.joblib']